In [149]:
import sys, os
import pandas as pd
from utils.misc import cols_to_front
from pathlib import Path
from datetime import datetime
import shutil
import nbformat
from nbconvert.preprocessors import ExecutePreprocessor
import yaml
with open('config/config.yaml', 'r') as file:
    config = yaml.safe_load(file)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")

In [150]:
exp_name = f"{config['exp_name']}_{timestamp}"   # your experiment name
RUN_DIR = Path(config["run_dir"]) /exp_name
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(RUN_DIR)

runs\flavor_match_2026-03-02_12-06


In [152]:
# Replacement string
target_str = 'REDUCE_DIM'
replacement = "config['reduce_dim']"

# -----------------------------
# Iterate over notebooks
# -----------------------------
notebooks = Path(".").glob("*.ipynb")

for nb_path in notebooks:

    # print(f"Processing {nb_path.name}")

    nb = nbformat.read(nb_path, as_version=4)

    modified = False

    for cell in nb.cells:
        if cell.cell_type == "code":
            if target_str in cell.source:
                cell.source = cell.source.replace(target_str, replacement)
                modified = True

    if modified:
        nbformat.write(nb, nb_path)
        print(f"Updated {nb_path.name}")
    else:
        print(f"No change in {nb_path.name}")

Processing azureAPI.ipynb
No change in azureAPI.ipynb
Processing data_exploration.ipynb
Updated data_exploration.ipynb
Processing data_processing.ipynb
No change in data_processing.ipynb
Processing ingredients_matching.ipynb
No change in ingredients_matching.ipynb
Processing save_experiment.ipynb
No change in save_experiment.ipynb
Processing vectorize_ai_descriptors.ipynb
No change in vectorize_ai_descriptors.ipynb


In [139]:
# Run all the pipeline notebooks sequentially
NOTEBOOK_DIR = Path(".")
notebooks = [
    "data_processing.ipynb",
    'vectorize_ai_descriptors.ipynb',
    'ingredients_matching.ipynb',
    'data_exploration.ipynb',
]

# -----------------------------------
# Execute notebooks sequentially
# -----------------------------------
for nb_name in notebooks:
    nb_path = NOTEBOOK_DIR / nb_name
    print(f"\n🚀 Executing {nb_name}")

    with open(nb_path) as f:
        nb = nbformat.read(f, as_version=4)

    ep = ExecutePreprocessor(
        timeout=600,          # increase if long runs
        kernel_name="python3"
    )

    ep.preprocess(nb, {"metadata": {"path": NOTEBOOK_DIR}})

    # Overwrite notebook with executed outputs
    with open(nb_path, "w", encoding="utf-8") as f:
        nbformat.write(nb, f)

    print(f"✅ Finished {nb_name}")


🚀 Executing data_processing.ipynb
✅ Finished data_processing.ipynb

🚀 Executing vectorize_ai_descriptors.ipynb
✅ Finished vectorize_ai_descriptors.ipynb

🚀 Executing ingredients_matching.ipynb
✅ Finished ingredients_matching.ipynb

🚀 Executing data_exploration.ipynb
✅ Finished data_exploration.ipynb


In [140]:
data_files = [
    'data/merged_ai_descriptors_clean.csv',
    'data/local_df_ai_descriptors.csv',
    'data/merged_ai_descriptors_dummies_filtered.csv',
    'data/merged_ai_descriptors_dummies_full.csv',
    'data/merged_ai_descriptors_clean.csv',
    'data/merged_ai_descriptors.csv',
    'data/flavor_db_descriptors.csv',
    'data/local_df_ai_descriptors.csv',
    'data/all_descriptors.csv'
]

output_files = [
    'output/flavordb_ingredients_top3_matches.xlsx',
    'output/local_ingredients_top3_matches.xlsx',
    'output/cat_clustered_jaccard.png',
    'output/db_jaccard.png',
    'output/cat_tsne.png',
    'output/db_tsne.png',
    "output/cat_jaccard_sim_top3.xlsx",
    
]

In [141]:
# Save data files
data_dir = RUN_DIR / "data"
data_dir.mkdir(parents=True, exist_ok=True)

# copy files
for file in data_files:
    src = Path(file)
    dst = data_dir / src.name
    shutil.copy2(src, dst)

In [142]:
# Save output files
data_dir = RUN_DIR / "output"
data_dir.mkdir(parents=True, exist_ok=True)

# copy files
for file in output_files:
    src = Path(file)
    dst = data_dir / src.name
    shutil.copy2(src, dst)

# Save overview

In [143]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# -----------------------------
# Input files (your lists)
# -----------------------------
xlsx_files = [
    'output/local_ingredients_top3_matches.xlsx',
    'output/flavordb_ingredients_top3_matches.xlsx'
]

png_files = [
    'output/cat_tsne.png',
    'output/cat_clustered_jaccard.png',
]

cols_to_drop = ["raw_AI_output", "sub_category", "db", "category"]

df1 = pd.read_excel(xlsx_files[0])
df2 = pd.read_excel(xlsx_files[1])

# Drop unwanted columns
df1 = df1.drop(columns=cols_to_drop, errors="ignore")
df2 = df2.drop(columns=cols_to_drop, errors="ignore")

# Selection of rows
ings =['Macrocystis pyrifera', 'Morchella crassipes', 'Picea rubens', 'Allium canadense', 'Rhus glabra', 'Artemisia frigida']
df1 = df1[df1["Nom scientifique"].isin(ings)]   

ings = ['Rice', 'Cedar', 'Sage', 'Apple', "Celery", "Lemon"]
df2 = df2[df2["Nom scientifique"].isin(ings)]   

In [144]:
# Rename matches to common name
merged_df = pd.read_csv('data/local_df_ai_descriptors.csv')

# Create mapping dict
mapping = dict(zip(
    merged_df["Nom scientifique"],
    merged_df["Nom vernaculaire"]
))

# Apply mapping (preserve original if no match)
cols = ["top1_name", "top2_name", "top3_name"]

for col in cols:
    df2[col] = df2[col].map(mapping)

# Combine into one comma-separated column
df2["matches"] = (
    df2[cols]
    .astype(str)
    .apply(lambda row: ", ".join(row.dropna()), axis=1)
)

# Drop some columns to make things more readable
df1 = df1[['Nom scientifique', 'Nom vernaculaire', 'matches', 'descriptor']]
df2 = df2[['Nom scientifique', 'matches', 'descriptor']]


In [145]:
out_path = RUN_DIR / f"{exp_name}_{timestamp}_summary.png"

# -----------------------------
# Load PNG images
# -----------------------------
img1 = Image.open(png_files[0]).convert("RGB")
img2 = Image.open(png_files[1]).convert("RGB")

# -----------------------------
# Convert df.head() → image
# -----------------------------
def df_to_image(df, title="", dpi=300):
    df = df.astype(str)

    nrows, ncols = df.shape

    # -----------------------------
    # Compute column widths from content length
    # -----------------------------
    max_lens = [
        max(
            df[col].str.len().max(),
            len(col)
        )
        for col in df.columns
    ]

    # normalize widths
    total = sum(max_lens)
    col_widths = [l / total for l in max_lens]

    # scale figure size based on total characters
    fig_w = max(16, total * 0.20)
    fig_h = max(6, 1.2 * (nrows + 1))

    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
    ax.axis("off")

    if title:
        ax.set_title(
            title,
            fontsize=20,
            pad=20,
            fontweight="bold"
        )

    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        loc="center",
        cellLoc="left",
        colWidths=col_widths   # <-- KEY FIX
    )

    table.auto_set_font_size(False)
    table.set_fontsize(20)
    table.scale(1, 2.0)   # don't scale width blindly

    # header styling
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_fontsize(22)
            cell.set_text_props(weight='bold')

    tmp = RUN_DIR / "__tmp_df.png"
    fig.savefig(tmp, bbox_inches="tight", dpi=dpi)
    plt.close(fig)

    im = Image.open(tmp).convert("RGB")
    tmp.unlink()

    return im


# Convert to images
df_img1 = df_to_image(df1, title=Path(xlsx_files[0]).name, dpi=DPI)
df_img2 = df_to_image(df2, title=Path(xlsx_files[1]).name, dpi=DPI)

# -----------------------------
# Resize all to same width
# -----------------------------
def resize_to_width(im, w):
    h = int(im.size[1] * w / im.size[0])
    return im.resize((w, h), Image.LANCZOS)

panel_w = min(
    img1.size[0], img2.size[0],
    df_img1.size[0], df_img2.size[0]
)

img1 = resize_to_width(img1, panel_w)
img2 = resize_to_width(img2, panel_w)
df_img1 = resize_to_width(df_img1, panel_w*2)
df_img2 = resize_to_width(df_img2, panel_w*2)

# -----------------------------
# Create composite (2x2 grid)
# -----------------------------
gap = 40
pad = 60
bg = (255, 255, 255)


row2_h = max(df_img1.size[1], df_img2.size[1])
row1_h = max(img1.size[1], img2.size[1])

canvas_w = pad*2 + panel_w*2 + gap
canvas_h = (
    pad*2
    + row1_h
    + gap
    + df_img1.size[1]
    + gap
    + df_img2.size[1]
)

canvas = Image.new("RGB", (canvas_w, canvas_h), bg)

# -----------------------------
# Paste images
# -----------------------------

# Top row (plots side by side)
canvas.paste(img1, (pad, pad))
canvas.paste(img2, (pad + panel_w + gap, pad))

# First dataframe (full width, centered under plots)
y_df1 = pad + row1_h + gap
canvas.paste(df_img1, (pad, y_df1))

# Second dataframe stacked below
y_df2 = y_df1 + df_img1.size[1] + gap
canvas.paste(df_img2, (pad, y_df2))
# -----------------------------
# Save
# -----------------------------
out_path.parent.mkdir(exist_ok=True)
canvas.save(out_path, dpi=(DPI, DPI))

# Save summary to parent runs folder for easy comparisons
canvas.save(Path('runs')/f"{exp_name}_{timestamp}_summary.png", dpi=(DPI, DPI))

print(f"Saved composite image to: {out_path}")

Saved composite image to: runs\flavor_match_2026-03-02_11-41\flavor_match_2026-03-02_11-41_summary.png


# Searching

In [135]:
# kw = "iso"
# flavor_db.loc[flavor_db['Nom scientifique'].str.contains(kw)]